In [1]:
import json
from monty.json import MontyDecoder
import os
from pymatgen.core import Element

from utils_kga.statistical_analysis.analyze_ion_pairs import *

In [2]:
# Load edge-df
with open("data/dfs_of_magnetic_edge_information_0p5_zero-magmom_threshold.json") as f:
    dict_all_stats = json.load(f)
all_stats = {key: pd.DataFrame.from_dict(df) for key, df in dict_all_stats.items()}

# For metadata filtering and computation of occurrences in magnetic primitive cells
with open("data/unique_4559_cnfeat_MP_db_from_api.json") as f:
    df = json.load(f, cls=MontyDecoder)

plot_dir = "data"

In [3]:
# The automatically determined oxidation states might be "unphysical" in many cases, e.g., that some ions are not expected to display magnetic order
# Nevertheless, we are showing the results as determined by the automated BVA

custom_sort_by_electron_configuration = [
    ('Mo', 6), ('W', 6), 
    ('Ti', 3), ("V", 4),  ("Cr", 5), ("Nb", 4), ("Mo", 5), ('W', 5), ('Re', 6),
    ('Ti', 4), ("V", 3), ("Cr", 4), ('Mn', 5), ("Mo", 4), ("W", 4), ("Re", 5), ("Os", 6),
    ("V", 2), ("Cr", 3), ("Mn", 4), ("Ru", 5), ("Mo", 3), ("Os", 5), ('Ir', 6), 
    ("Cr", 2), ("Mn", 3), ("Fe", 4),
    ("Ru", 4), ("Ir", 5),
    ("Mn", 2), ("Fe", 3), ("Co", 4),
    ("Rh", 4), ("Ir", 4),
    ("Fe", 2), ("Co", 3), ('Ni', 4),
    ('Fe', 1), ("Co", 2), ("Ni", 3),
    ("Co", 1), ("Ni", 2), ('Cu', 3),
    ("Ni", 1), ("Cu", 2)
]

In [4]:
for ang_df in all_stats.values():
    ang_df["site_is_tm"] = ang_df["site_element"].apply(lambda el: Element(el).is_transition_metal)
    ang_df["site_to_is_tm"] = ang_df["site_to_element"].apply(lambda el: Element(el).is_transition_metal)
    ang_df["ligand_el_set"] = ang_df["ligand_elements"].apply(lambda ls: set(ls))



In [5]:
write_mode = "w"
ba_threshold = 150
normalize_bool = False
normalize_string = "absolute occurrences"

missed_ions = set()

data_string = f"corner-connected. TM octahedra at both nodes"
print(data_string)
occus = {"afm": {}, "fm": {}}
num_structures = {}  # track in how many structures ion pair is found with these conditions

for md_id, ang_df in all_stats.items():
    subdf = ang_df.loc[(ang_df["site_is_tm"])  & (ang_df["site_to_is_tm"])]
    subdf = subdf.loc[subdf["connectivity"]=="corner"]
    subdf = subdf.loc[(subdf["site_ce"]=="O:6")
            & (subdf["site_to_ce"]=="O:6")]

    if not subdf.empty:
        n_lattice_points = df.at[md_id, "n_mag_lattice_points"]
        for mag_type, condition in zip(["afm", "fm", ],
                                        [(subdf["spin_angle"]>170), (subdf["spin_angle"]<10)]):
            type_df = subdf.loc[condition]
            if not type_df.empty:
                abs_ion_pairs = get_ion_pair_occus(df=type_df, n_lattice_points=n_lattice_points, normalize_bool=normalize_bool)

                for k, v in abs_ion_pairs.items():
                    # Assert no detected ions are missing 
                    try:
                        assert (k[0], k[1]) in custom_sort_by_electron_configuration, str((k[0], k[1]))
                        assert (k[2], k[3]) in custom_sort_by_electron_configuration, str((k[2], k[3]))
                    except AssertionError as e:
                        missed_ions.add(str(e))
                    if k in occus[mag_type]:
                        occus[mag_type][k] += v
                    else:
                        occus[mag_type][k] = v

                    if k in num_structures:
                        num_structures[k].add(md_id)
                    else:
                        num_structures[k] = {md_id}



print(occus)


# Plot heatmap of FM/AFM ratio per ion pair (logarithmic, diverging color scale)
ratio_fig = plot_ion_pair_occurrence_ratio(occus=occus, ion_list=custom_sort_by_electron_configuration)
ratio_fig.update_layout(title=dict(
    text=f"Logarithmic plot of FM/AFM ratios, {data_string}, {normalize_string}",
    font=dict(size=10)))
ratio_fig.update_xaxes(dict(
        tickfont=dict(size=14)))
ratio_fig.update_yaxes(dict(
    tickfont=dict(size=14)))
#ratio_fig.show()
ratio_fig.write_html(os.path.join(plot_dir, f"log_plot_afm-fm-ratio_ion_pair_{normalize_string.replace(' ', '_')}_{data_string.replace(' ', '_').replace('°', 'deg')}.html"))

close_to_1_tracker, all_tracker = set(), set()
print("FM to AFM ratio  :")

for ip in occus["fm"]:
    ip_string = "".join(sorted([str(ip[0])+str(ip[1]), str(ip[2])+str(ip[3])]))
    if ip in occus["afm"]:
        print(ip, round(occus["fm"][ip] / occus["afm"][ip], 2), round(log10(occus["fm"][ip] / occus["afm"][ip]), 2), len(num_structures[ip]))
        if abs(log10(occus["fm"][ip] / occus["afm"][ip])) < 0.5:
            close_to_1_tracker.add(ip_string)
        all_tracker.add(ip_string)
    else:
        print(ip, "-- (only FM)", len(num_structures[ip]))
        all_tracker.add(ip_string)
for ip in occus["afm"]:
    ip_string = "".join(sorted([str(ip[0])+str(ip[1]), str(ip[2])+str(ip[3])]))
    if not ip in occus["fm"]:
        print(ip, "-- (only AFM)", len(num_structures[ip]))
        all_tracker.add(ip_string)

print(len(close_to_1_tracker), len(all_tracker), len(close_to_1_tracker)/len(all_tracker))


n = set()
for s in num_structures.values():
    n.update(s)
print("Num structures: ", len(n))
print("missed: ", missed_ions)

# Display num structures in similar plot
numstructfig = plot_ion_pair_occurrences(occus={"num_structs": {k: len(v) for k, v in num_structures.items()}}, 
                                         log=False, ion_list=custom_sort_by_electron_configuration)["num_structs"]
numstructfig.update_layout(title=dict(
    text=f"Number of structures, {data_string}, {normalize_string}",
    font=dict(size=10)))
numstructfig.update_xaxes(dict(
        tickfont=dict(size=14)))
numstructfig.update_yaxes(dict(
    tickfont=dict(size=14)))
#numstructfig.show()
numstructfig.write_html(os.path.join(plot_dir, f"num_structures_ion_pair_{data_string.replace(' ', '_').replace('°', 'deg')}.html"))

corner-connected. TM octahedra at both nodes
{'afm': {('Mn', 3, 'Mn', 3): 220.0, ('Mn', 4, 'Mn', 3): 48.0, ('Mn', 3, 'Mn', 4): 48.0, ('Mn', 4, 'Mn', 4): 238.0, ('Cr', 2, 'W', 6): 80.0, ('W', 6, 'Cr', 2): 80.0, ('Cr', 3, 'W', 5): 16.0, ('W', 5, 'Cr', 3): 16.0, ('V', 3, 'W', 5): 16.0, ('W', 5, 'V', 3): 16.0, ('Mn', 2, 'W', 6): 40.0, ('W', 6, 'Mn', 2): 40.0, ('Fe', 2, 'W', 6): 12.0, ('W', 6, 'Fe', 2): 12.0, ('Mn', 2, 'Mn', 2): 916.0, ('Fe', 4, 'Fe', 4): 20.0, ('Cr', 4, 'Cr', 4): 104.0, ('Fe', 1, 'Fe', 1): 36.0, ('V', 4, 'Fe', 3): 40.0, ('Fe', 3, 'V', 4): 40.0, ('Fe', 3, 'W', 4): 52.0, ('W', 4, 'Fe', 3): 52.0, ('Mn', 4, 'Fe', 3): 44.0, ('Fe', 3, 'Mn', 4): 44.0, ('V', 2, 'V', 2): 90.0, ('V', 2, 'V', 3): 42.0, ('V', 3, 'V', 2): 42.0, ('V', 3, 'V', 3): 286.0, ('Cr', 3, 'Cr', 3): 410.0, ('Fe', 3, 'Fe', 3): 376.0, ('Mn', 2, 'Mn', 3): 172.0, ('Mn', 3, 'Mn', 2): 172.0, ('V', 4, 'V', 4): 152.0, ('Ni', 3, 'Ir', 5): 32.0, ('Ir', 5, 'Ni', 3): 32.0, ('V', 4, 'Cr', 3): 28.0, ('Cr', 3, 'V', 4): 28.0, ('

In [6]:
# Repeat analysis considering only oxygen edges

write_mode = "w"
ba_threshold = 150
normalize_bool = False
normalize_string = "absolute occurrences"

missed_ions = set()

data_string = f"corner-connected TM octahedra at both nodes only oxygen edges"
print(data_string)
occus = {"afm": {}, "fm": {}}
num_structures = {}  # track in how many structures ion pair is found with these conditions

for md_id, ang_df in all_stats.items():
    subdf = ang_df.loc[(ang_df["site_is_tm"])  & (ang_df["site_to_is_tm"])]
    subdf = subdf.loc[subdf["connectivity"]=="corner"]
    subdf = subdf.loc[(subdf["site_ce"]=="O:6")
            & (subdf["site_to_ce"]=="O:6")]
    subdf = subdf.loc[subdf["ligand_el_set"]=={"O"}]

    if not subdf.empty:
        n_lattice_points = df.at[md_id, "n_mag_lattice_points"]
        for mag_type, condition in zip(["afm", "fm", ],
                                        [(subdf["spin_angle"]>170), (subdf["spin_angle"]<10)]):
            type_df = subdf.loc[condition]
            if not type_df.empty:
                abs_ion_pairs = get_ion_pair_occus(df=type_df, n_lattice_points=n_lattice_points, normalize_bool=normalize_bool)

                for k, v in abs_ion_pairs.items():
                    # Assert no detected ions are missing 
                    try:
                        assert (k[0], k[1]) in custom_sort_by_electron_configuration, str((k[0], k[1]))
                        assert (k[2], k[3]) in custom_sort_by_electron_configuration, str((k[2], k[3]))
                    except AssertionError as e:
                        missed_ions.add(str(e))
                    if k in occus[mag_type]:
                        occus[mag_type][k] += v
                    else:
                        occus[mag_type][k] = v

                    if k in num_structures:
                        num_structures[k].add(md_id)
                    else:
                        num_structures[k] = {md_id}



print(occus)


# Plot heatmap of FM/AFM ratio per ion pair (logarithmic, diverging color scale)
ratio_fig = plot_ion_pair_occurrence_ratio(occus=occus, ion_list=custom_sort_by_electron_configuration)
ratio_fig.update_layout(title=dict(
    text=f"Logarithmic plot of FM/AFM ratios, {data_string}, {normalize_string}",
    font=dict(size=10)))
ratio_fig.update_xaxes(dict(
        tickfont=dict(size=14)))
ratio_fig.update_yaxes(dict(
    tickfont=dict(size=14)))
# ratio_fig.show()
ratio_fig.write_html(os.path.join(plot_dir, f"log_plot_afm-fm-ratio_ion_pair_{normalize_string.replace(' ', '_')}_{data_string.replace(' ', '_').replace('°', 'deg')}.html"))

close_to_1_tracker, all_tracker = set(), set()
print("FM to AFM ratio  :")

for ip in occus["fm"]:
    ip_string = "".join(sorted([str(ip[0])+str(ip[1]), str(ip[2])+str(ip[3])]))
    if ip in occus["afm"]:
        print(ip, round(occus["fm"][ip] / occus["afm"][ip], 2), round(log10(occus["fm"][ip] / occus["afm"][ip]), 2), len(num_structures[ip]))
        if abs(log10(occus["fm"][ip] / occus["afm"][ip])) < 0.5:
            close_to_1_tracker.add(ip_string)
        all_tracker.add(ip_string)
    else:
        print(ip, "-- (only FM)", len(num_structures[ip]))
        all_tracker.add(ip_string)
for ip in occus["afm"]:
    ip_string = "".join(sorted([str(ip[0])+str(ip[1]), str(ip[2])+str(ip[3])]))
    if not ip in occus["fm"]:
        print(ip, "-- (only AFM)", len(num_structures[ip]))
        all_tracker.add(ip_string)

print(len(close_to_1_tracker), len(all_tracker), len(close_to_1_tracker)/len(all_tracker))


n = set()
for s in num_structures.values():
    n.update(s)
print("Num structures: ", len(n))
print("missed: ", missed_ions)

# Display num structures in similar plot
numstructfig = plot_ion_pair_occurrences(occus={"num_structs": {k: len(v) for k, v in num_structures.items()}}, 
                                         log=False, ion_list=custom_sort_by_electron_configuration)["num_structs"]
numstructfig.update_layout(title=dict(
    text=f"Number of structures, {data_string}, {normalize_string}",
    font=dict(size=10)))
numstructfig.update_xaxes(dict(
        tickfont=dict(size=14)))
numstructfig.update_yaxes(dict(
    tickfont=dict(size=14)))
#numstructfig.show()
numstructfig.write_html(os.path.join(plot_dir, f"num_structures_ion_pair_{data_string.replace(' ', '_').replace('°', 'deg')}.html"))

corner-connected TM octahedra at both nodes only oxygen edges
{'afm': {('Mn', 3, 'Mn', 3): 164.0, ('Mn', 4, 'Mn', 3): 44.0, ('Mn', 3, 'Mn', 4): 44.0, ('Mn', 4, 'Mn', 4): 238.0, ('Cr', 2, 'W', 6): 80.0, ('W', 6, 'Cr', 2): 80.0, ('Cr', 3, 'W', 5): 16.0, ('W', 5, 'Cr', 3): 16.0, ('V', 3, 'W', 5): 16.0, ('W', 5, 'V', 3): 16.0, ('Mn', 2, 'W', 6): 40.0, ('W', 6, 'Mn', 2): 40.0, ('Fe', 2, 'W', 6): 12.0, ('W', 6, 'Fe', 2): 12.0, ('Mn', 2, 'Mn', 2): 744.0, ('Fe', 4, 'Fe', 4): 20.0, ('Cr', 4, 'Cr', 4): 104.0, ('Fe', 1, 'Fe', 1): 36.0, ('V', 4, 'Fe', 3): 32.0, ('Fe', 3, 'V', 4): 32.0, ('Fe', 3, 'W', 4): 52.0, ('W', 4, 'Fe', 3): 52.0, ('Mn', 4, 'Fe', 3): 44.0, ('Fe', 3, 'Mn', 4): 44.0, ('V', 2, 'V', 2): 86.0, ('V', 2, 'V', 3): 42.0, ('V', 3, 'V', 2): 42.0, ('V', 3, 'V', 3): 234.0, ('Cr', 3, 'Cr', 3): 398.0, ('Fe', 3, 'Fe', 3): 306.0, ('Mn', 2, 'Mn', 3): 112.0, ('Mn', 3, 'Mn', 2): 112.0, ('V', 4, 'V', 4): 104.0, ('Ni', 3, 'Ir', 5): 32.0, ('Ir', 5, 'Ni', 3): 32.0, ('V', 4, 'Cr', 3): 28.0, ('Cr', 3, 